# SFVP AI Clone — Notebook 05: Fine-Tune Voice
**Run this when:** you have 5+ minutes of total voice sample audio collected

Fine-tuning teaches F5-TTS Shennel's specific voice at a deeper level — her unique rhythm, tone, inflections, and expressiveness. Zero-shot cloning (Notebook 01) is good; fine-tuned cloning is **her**.

**Each fine-tune session improves as you add more samples.** Run it again whenever you've added significant new recordings.

**Time:** ~20-30 minutes on T4 GPU
**Cost:** $0 (free T4)

**Drive structure needed:**
```
My Drive/SFVP_Clone/
├── voice_samples/       ← all your voice recordings (.wav/.mp3/.m4a)
└── fine_tuned_model/    ← trained checkpoint saved here (created automatically)
```

In [ ]:
# CELL 1 — Install F5-TTS with training support
!pip install git+https://github.com/SWivid/F5-TTS.git -q
!pip install soundfile librosa -q
!apt-get install -y ffmpeg -q
!pip install faster-whisper -q  # for auto-transcription of your voice samples
print('Installed.')

In [ ]:
# CELL 2 — Mount Drive + check samples
from google.colab import drive
import os

drive.mount('/content/drive')
DRIVE_BASE = '/content/drive/MyDrive/SFVP_Clone'
SAMPLES_DIR = f'{DRIVE_BASE}/voice_samples'
MODEL_SAVE_DIR = f'{DRIVE_BASE}/fine_tuned_model'
os.makedirs(MODEL_SAVE_DIR, exist_ok=True)

samples = [f for f in os.listdir(SAMPLES_DIR) if f.endswith(('.wav','.mp3','.m4a','.aac'))]
print(f'Found {len(samples)} voice sample(s):')
for s in samples:
    print(f'  {s}')

if len(samples) < 3:
    print('\nWARNING: Fine-tuning works best with 5+ minutes of audio.')
    print('You can proceed with fewer samples but quality will be limited.')
    print('Keep recording and re-run this notebook as you add more.')

In [ ]:
# CELL 3 — Convert all samples to 24kHz WAV + auto-transcribe
# This cell prepares the training dataset automatically — no manual labeling needed.

import subprocess
import json
from faster_whisper import WhisperModel

TRAIN_DIR = '/content/sfvp_train'
WAVS_DIR = f'{TRAIN_DIR}/wavs'
os.makedirs(WAVS_DIR, exist_ok=True)

print('Loading Whisper for transcription (small model, ~150MB)...')
whisper = WhisperModel('small', device='cuda', compute_type='float16')

metadata = []  # list of {audio_path, text} pairs

for i, sample_file in enumerate(samples):
    src = f'{SAMPLES_DIR}/{sample_file}'
    wav_out = f'{WAVS_DIR}/sample_{i:03d}.wav'
    
    # Convert to 24kHz mono WAV
    subprocess.run(
        ['ffmpeg', '-y', '-i', src, '-ar', '24000', '-ac', '1', wav_out],
        capture_output=True
    )
    
    # Auto-transcribe
    segments, _ = whisper.transcribe(wav_out, language='en')
    text = ' '.join(seg.text.strip() for seg in segments).strip()
    
    metadata.append({'audio_path': wav_out, 'text': text})
    print(f'  [{i+1}/{len(samples)}] {sample_file}: "{text[:60]}..."' if len(text) > 60 else f'  [{i+1}/{len(samples)}] {sample_file}: "{text}"')

# Save metadata CSV for F5-TTS training format
metadata_path = f'{TRAIN_DIR}/metadata.csv'
with open(metadata_path, 'w', encoding='utf-8') as f:
    f.write('audio_path|text\n')
    for m in metadata:
        f.write(f'{m["audio_path"]}|{m["text"]}\n')

print(f'\nDataset ready: {len(metadata)} samples')

# Compute total duration
total_dur = 0
for m in metadata:
    r = subprocess.run(['ffprobe','-v','quiet','-show_entries','format=duration','-of','csv=p=0',m['audio_path']],
                        capture_output=True, text=True)
    try:
        total_dur += float(r.stdout.strip())
    except:
        pass
print(f'Total audio: {total_dur/60:.1f} minutes')

In [ ]:
# CELL 4 — CONFIGURE fine-tuning settings

# Training steps — more = better quality but longer training
# With <5 min audio: 500 steps
# With 5-15 min audio: 1000 steps
# With 15+ min audio: 2000 steps
TRAIN_STEPS = 1000  # <-- ADJUST BASED ON YOUR AUDIO AMOUNT

BATCH_SIZE = 4       # T4 handles 4 fine
LEARNING_RATE = 1e-4
SAVE_CHECKPOINT_EVERY = 250  # saves checkpoint every N steps

print(f'Fine-tune config:')
print(f'  Steps:         {TRAIN_STEPS}')
print(f'  Batch size:    {BATCH_SIZE}')
print(f'  Learning rate: {LEARNING_RATE}')
print(f'  Estimated time: ~{TRAIN_STEPS * 1.5 / 60:.0f}-{TRAIN_STEPS * 2.5 / 60:.0f} minutes on T4')

In [ ]:
# CELL 5 — Download base F5-TTS model to fine-tune from
from huggingface_hub import hf_hub_download

BASE_MODEL_DIR = '/content/f5tts_base'
os.makedirs(BASE_MODEL_DIR, exist_ok=True)

print('Downloading F5-TTS base model...')
hf_hub_download(
    repo_id='SWivid/F5-TTS',
    filename='F5TTS_Base/model_1200000.safetensors',
    local_dir=BASE_MODEL_DIR
)
hf_hub_download(
    repo_id='SWivid/F5-TTS',
    filename='F5TTS_Base/config.yaml',
    local_dir=BASE_MODEL_DIR
)
print('Base model ready.')

In [ ]:
# CELL 6 — Run fine-tuning
import subprocess

checkpoint_dir = '/content/sfvp_checkpoints'
os.makedirs(checkpoint_dir, exist_ok=True)

cmd = [
    'python', '-m', 'f5_tts.train.finetune_cli',
    '--exp_name', 'SFVP_Shennel',
    '--learning_rate', str(LEARNING_RATE),
    '--batch_size_per_gpu', str(BATCH_SIZE),
    '--batch_size_type', 'sample',
    '--max_samples', '64',
    '--grad_accumulation_steps', '1',
    '--max_grad_norm', '1.0',
    '--epochs', '100',
    '--num_warmup_updates', '50',
    '--save_per_updates', str(SAVE_CHECKPOINT_EVERY),
    '--last_per_updates', str(SAVE_CHECKPOINT_EVERY),
    '--dataset_name', 'SFVP_Voice',
    '--model', 'F5TTS_Base',
    '--pretrain_model_path', f'{BASE_MODEL_DIR}/F5TTS_Base/model_1200000.safetensors',
    '--tokenizer', 'pinyin',
    '--tokenizer_path', 'None',
    '--logger', 'tensorboard',
    '--log_samples', 'true',
]

# Write dataset config
import yaml
dataset_config = {
    'datasets': [{
        'name': 'SFVP_Voice',
        'metadata_path': metadata_path,
        'sample_rate': 24000,
    }]
}
with open('/content/dataset_config.yaml', 'w') as f:
    yaml.dump(dataset_config, f)

print(f'Starting fine-tune ({TRAIN_STEPS} steps)...')
print('Progress will print below. This takes 20-30 min on T4.')
print('Do NOT close this tab — Colab will stop the training.')
print()

result = subprocess.run(cmd, text=True)

if result.returncode == 0:
    print('Fine-tuning complete!')
else:
    print('Training ended (may be OK — check output above)')

In [ ]:
# CELL 7 — Save trained model to Google Drive
import shutil
import glob

# Find the latest checkpoint
checkpoints = sorted(glob.glob('/content/ckpts/SFVP_Shennel/*.pt') +
                     glob.glob('/content/ckpts/SFVP_Shennel/*.safetensors'))

if checkpoints:
    latest = checkpoints[-1]
    dest = f'{MODEL_SAVE_DIR}/shennel_voice_checkpoint.pt'
    shutil.copy(latest, dest)
    print(f'Model saved to Drive: {dest}')
    print()
    print('To use this fine-tuned model in Notebook 01 or 00:')
    print(f'  Set FINE_TUNED_MODEL_PATH = "{dest}" in the config cell')
else:
    print('No checkpoint found — check training output above')
    print('If training ran for at least 250 steps, check /content/ckpts/')

In [ ]:
# CELL 8 — Test the fine-tuned model
from f5_tts.api import F5TTS
from IPython.display import Audio, display
import shutil

TEST_SCRIPT = "Real talk — Sactown's Finest is where Sacramento comes to get seen. Custom vinyl, banners, print. We built this city's brand. You're next."
REF_SAMPLE = samples[0]  # uses first voice sample as reference

# Convert ref to WAV
ref_wav = '/content/test_ref.wav'
subprocess.run(['ffmpeg','-y','-i',f'{SAMPLES_DIR}/{REF_SAMPLE}','-t','10','-ar','24000','-ac','1',ref_wav],
               capture_output=True)

tts = F5TTS(ckpt_file=dest)  # loads fine-tuned checkpoint

print('Generating test audio with fine-tuned model...')
wav, sr, _ = tts.infer(
    ref_file=ref_wav,
    ref_text='',
    gen_text=TEST_SCRIPT,
    file_wave='/content/fine_tuned_test.wav',
    remove_silence=True,
)

print('Fine-tuned model output:')
display(Audio('/content/fine_tuned_test.wav'))
print()
print('Compare this to Notebook 01 output (zero-shot) — the fine-tuned version should sound more like her.')